# Selection Efficiency audit: full rerun

1. Upload the whole `se_audit` folder to Google Drive at `MyDrive/se_audit`.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run the cells top to bottom.

Every step is resumable. After a disconnect, rerun cells 1–2 and then the step you were on; finished items are skipped. Outputs go to `MyDrive/se_audit_out`, so nothing is lost when the runtime resets.

In [ ]:
# 1. Mount Drive and point outputs at it
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['SE_OUT'] = '/content/drive/MyDrive/se_audit_out'
%cd /content/drive/MyDrive/se_audit
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Dependencies (torch is preinstalled on Colab)
!pip install -q 'transformers>=4.43' 'sentence-transformers>=2.6' scipy

In [ ]:
# 3. Build the four splits (seconds)
!python data.py

## Generation
One cell per model, each about 40–80 min on a T4. If a cell is interrupted, just run it again.

In [ ]:
!python generate.py --model qwen1.5b

In [ ]:
!python generate.py --model qwen3b

In [ ]:
!python generate.py --model phi2

In [ ]:
# capability-floor appendix case
!python generate.py --model smollm2

## Scoring and analysis

In [ ]:
# NLI edge probabilities (RoBERTa + BART) and trace cosine, about 10 min
!python score.py

In [ ]:
# All tables + claims check (CPU, seconds)
!python analyze.py

In [ ]:
# Look at the recomputed claims and one table
from IPython.display import Markdown, display
out = os.environ['SE_OUT'] + '/tables'
display(Markdown(open(out + '/claims.md').read()))
print(open(out + '/tab_methods_full.tex').read())

In [ ]:
# Bundle the tables to download (and to send back for the paper edits)
!cd $SE_OUT && zip -qr tables.zip tables && ls -la tables.zip
from google.colab import files
files.download(os.environ['SE_OUT'] + '/tables.zip')